# Gemma-3 Multilingual Fine-tuning with Stratified Validation

**Complete training pipeline for Cebuano, Ilocano, and Tagalog:**
- Language stratification (80/10/10 splits)
- Continuous validation with language-aware BERTScore
- Early stopping with best checkpoint saving
- Google Drive integration for persistent storage

📌 **Setup Instructions:**
1. Upload your JSONL file to Colab: `Completed_combined_and_shuffled_datasets.jsonl`
2. Run each cell sequentially
3. Monitor GPU memory and outputs between sections

## Cell 1: Install Required Packages

In [ ]:
# Install required packages for Colab
import subprocess
import sys

print("Installing required packages...")
packages = [
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    'transformers',
    'datasets',
    'bert-score',
    'scikit-learn',
    'pandas',
    'matplotlib',
    'seaborn'
]

for package in packages:
    print(f"\nInstalling {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✓ All packages installed successfully!")

## Cell 2: Import Libraries and Setup Google Drive

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ML/Training libraries
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, standardize_data_formats, train_on_responses_only
from trl import SFTTrainer, SFTConfig

# Google Colab setup
from google.colab import drive, files

print("✓ All libraries imported successfully!")

# Mount Google Drive
print("\nMounting Google Drive...")
drive.mount('/content/drive', force_remount=True)
BASE_DRIVE_PATH = "/content/drive/MyDrive/gemma_3_multilingual"
os.makedirs(BASE_DRIVE_PATH, exist_ok=True)
print(f"✓ Google Drive mounted. Base path: {BASE_DRIVE_PATH}")

# Check GPU
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Cell 3: Load and Explore JSONL Dataset

In [ ]:
def load_and_explore_dataset(jsonl_path: str = 'Completed_combined_and_shuffled_datasets.jsonl') -> Dataset:
    """Load JSONL dataset and explore structure."""
    print(f"Loading JSONL dataset from: {jsonl_path}")

    try:
        dataset = load_dataset('json', data_files=jsonl_path, split='train')
        print(f"✓ Dataset loaded successfully!")
        print(f"  Total examples: {len(dataset)}")
        print(f"  Dataset columns: {dataset.column_names}")

        # Display sample
        print("\n--- Sample Examples ---")
        for i in range(min(3, len(dataset))):
            print(f"\nExample {i+1}:")
            example = dataset[i]
            for key, value in example.items():
                if isinstance(value, str) and len(value) > 100:
                    print(f"  {key}: {value[:100]}...")
                else:
                    print(f"  {key}: {value}")

        return dataset
    except FileNotFoundError:
        print(f"Error: File '{jsonl_path}' not found!")
        print("Please upload the JSONL file to Colab first.")
        return None

# Load dataset
dataset = load_and_explore_dataset()

if dataset is None:
    print("\n⚠️  Upload your JSONL file and try again.")

## Cell 4: Language Detection and Stratified Dataset Splitting

In [ ]:
# Language detection patterns
LANGUAGE_PATTERNS = {
    'cebuano': ['ang', 'sa', 'ni', 'ug', 'na', 'kay', 'dali', 'ubos', 'daghan', 'maayo'],
    'ilocano': ['ti', 'dagiti', 'ket', 'ay', 'a', 'ni', 'isu', 'babaen', 'no', 'man'],
    'tagalog': ['ang', 'sa', 'ng', 'ay', 'na', 'para', 'pero', 'kung', 'ito', 'yan']
}

def detect_language_from_text(text: str) -> str:
    """Detect language based on common word patterns."""
    if not isinstance(text, str):
        return 'unknown'

    text_lower = text.lower()

    # Try to extract language from metadata
    if '[LANGUAGE]' in text or 'language' in text:
        if 'cebuano' in text_lower:
            return 'cebuano'
        elif 'ilocano' in text_lower:
            return 'ilocano'
        elif 'tagalog' in text_lower:
            return 'tagalog'

    # Count language-specific word occurrences
    language_scores = {}
    for lang, words in LANGUAGE_PATTERNS.items():
        score = sum(1 for word in words if word in text_lower)
        language_scores[lang] = score

    if max(language_scores.values()) > 0:
        return max(language_scores, key=language_scores.get)
    else:
        return 'mixed'

def extract_language_stratification(dataset: Dataset) -> Tuple[List[str], List[int]]:
    """Extract language for each example."""
    languages = []
    language_counts = Counter()

    print("Detecting language for each example...")
    for example in dataset:
        if 'conversations' in example:
            text = str(example['conversations'])
        elif 'text' in example:
            text = example['text']
        else:
            text = str(example)

        lang = detect_language_from_text(text)
        languages.append(lang)
        language_counts[lang] += 1

    print("\n--- Language Distribution ---")
    for lang, count in language_counts.most_common():
        percentage = (count / len(languages)) * 100
        print(f"  {lang}: {count} ({percentage:.1f}%)")

    language_to_label = {lang: idx for idx, lang in enumerate(language_counts.keys())}
    stratify_labels = [language_to_label[lang] for lang in languages]

    return languages, stratify_labels

def perform_stratified_split(dataset: Dataset, stratify_labels: List[int]):
    """Perform stratified 80/10/10 train/val/test split."""
    print("\nPerforming stratified train/val/test split...")

    indices = np.arange(len(dataset))
    train_indices, temp_indices = train_test_split(
        indices,
        test_size=0.2,
        stratify=[stratify_labels[i] for i in indices],
        random_state=42
    )

    val_indices, test_indices = train_test_split(
        temp_indices,
        test_size=0.5,
        stratify=[stratify_labels[i] for i in temp_indices],
        random_state=42
    )

    train_dataset = dataset.select(train_indices.tolist())
    val_dataset = dataset.select(val_indices.tolist())
    test_dataset = dataset.select(test_indices.tolist())

    print(f"\n--- Split Sizes ---")
    print(f"  Training:   {len(train_dataset)} ({len(train_dataset)/len(dataset)*100:.1f}%)")
    print(f"  Validation: {len(val_dataset)} ({len(val_dataset)/len(dataset)*100:.1f}%)")
    print(f"  Test:       {len(test_dataset)} ({len(test_dataset)/len(dataset)*100:.1f}%)")

    return train_dataset, val_dataset, test_dataset

# Perform language detection and splitting
if dataset is not None:
    languages, stratify_labels = extract_language_stratification(dataset)
    train_dataset, val_dataset, test_dataset = perform_stratified_split(dataset, stratify_labels)
    print(f"\n✓ Dataset splitting completed successfully!")

## Cell 5: Configure Model and LoRA Adapters

In [ ]:
print("Loading Gemma-3 4B model with 4-bit quantization...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

print("✓ Model loaded successfully!")

# Apply LoRA adapters
print("\nApplying LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✓ LoRA adapters applied!")

# Setup tokenizer with Gemma-3 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

print("✓ Tokenizer configured!")

## Cell 6: Standardize and Format Datasets

In [ ]:
# Standardize dataset format
print("Standardizing dataset format...")
train_dataset = standardize_data_formats(train_dataset)
val_dataset = standardize_data_formats(val_dataset)
test_dataset = standardize_data_formats(test_dataset)
print("✓ Dataset format standardized!")

# Apply chat template formatting
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(
        convo,
        tokenize=False,
        add_generation_prompt=False
    ).removeprefix('<bos>') for convo in convos]
    return {"text": texts}

print("\nApplying chat template to datasets...")
train_dataset = train_dataset.map(formatting_prompts_func, batched=True, desc="Train")
val_dataset = val_dataset.map(formatting_prompts_func, batched=True, desc="Validation")
test_dataset = test_dataset.map(formatting_prompts_func, batched=True, desc="Test")
print("✓ Chat template applied!")

# Display sample
print("\n--- Sample formatted conversation ---")
print(train_dataset[0]["text"][:200] + "...")

## Cell 7: Setup Language-Aware BERTScore Evaluator

In [ ]:
class LanguageAwareBERTScoreEvaluator:
    """Evaluates using language-specific BERT models."""

    def __init__(self):
        self.language_to_model = {
            'cebuano': 'DOST-ASTI/bert-ceb-cased',
            'tagalog': 'DOST-ASTI/bert-ceb-cased',
            'ilocano': 'bert-base-multilingual-cased',
            'mixed': 'bert-base-multilingual-cased',
            'unknown': 'bert-base-multilingual-cased'
        }
        self.loaded_models = {}

    def get_model_and_tokenizer(self, language: str):
        """Load language-specific BERT model."""
        if language not in self.loaded_models:
            model_name = self.language_to_model.get(language, 'bert-base-multilingual-cased')
            try:
                tokenizer = AutoTokenizer.from_pretrained(model_name)
                model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
                model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
                model.eval()
                self.loaded_models[language] = (model, tokenizer)
                print(f"  ✓ Loaded model for {language}: {model_name}")
            except Exception as e:
                print(f"  ⚠️  Warning: Could not load {model_name}. Using multilingual model.")
                tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
                model = AutoModel.from_pretrained('bert-base-multilingual-cased')
                model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
                model.eval()
                self.loaded_models[language] = (model, tokenizer)

        return self.loaded_models[language]

    def compute_weighted_similarity(self, pred: str, ref: str, language: str) -> float:
        """Compute weighted BERTScore."""
        model, tokenizer = self.get_model_and_tokenizer(language)

        try:
            with torch.no_grad():
                pred_encoded = tokenizer.encode(pred, return_tensors='pt').to(model.device)
                ref_encoded = tokenizer.encode(ref, return_tensors='pt').to(model.device)

                pred_embedding = model(pred_encoded)[0].mean(dim=1)
                ref_embedding = model(ref_encoded)[0].mean(dim=1)

            similarity = torch.nn.functional.cosine_similarity(pred_embedding, ref_embedding)
            return float(similarity.cpu()[0])
        except Exception as e:
            return 0.0

# Initialize evaluator
print("Initializing Language-aware BERTScore evaluator...")
evaluator = LanguageAwareBERTScoreEvaluator()
print("✓ Evaluator initialized!")

## Cell 8: Setup Output Directories and Training Configuration

In [ ]:
# Create output directories for checkpoints, models, and metrics
checkpoint_dir = os.path.join(BASE_DRIVE_PATH, 'checkpoints')
best_model_dir = os.path.join(BASE_DRIVE_PATH, 'best_model')
metrics_dir = os.path.join(BASE_DRIVE_PATH, 'metrics')

for dir_path in [checkpoint_dir, best_model_dir, metrics_dir]:
    os.makedirs(dir_path, exist_ok=True)

print(f"✓ Output directories created!")
print(f"  Checkpoints: {checkpoint_dir}")
print(f"  Best model: {best_model_dir}")
print(f"  Metrics: {metrics_dir}")

# ============================================================
# TRAINING CONFIGURATION - All hyperparameters here
# ============================================================
training_config = {
    # Dataset
    'dataset_text_field': 'text',

    # Batch sizes (adjust based on GPU memory)
    'per_device_train_batch_size': 2,
    'per_device_eval_batch_size': 4,
    'gradient_accumulation_steps': 4,

    # Learning rate schedule
    'warmup_steps': 50,
    'num_train_epochs': 3,
    'learning_rate': 2e-4,
    'lr_scheduler_type': 'cosine',

    # Optimizer
    'optim': 'adamw_8bit',
    'weight_decay': 0.1,

    # Logging
    'logging_steps': 10,
    'log_level': 'info',

    # Evaluation
    'eval_strategy': 'epoch',        # Evaluate every epoch
    'save_strategy': 'epoch',        # Save checkpoint every epoch
    'load_best_model_at_end': True,  # Load best checkpoint after training
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,

    # Paths
    'output_dir': checkpoint_dir,
    'seed': 3407,

    # Misc
    'eval_accumulation_steps': 4,
    'report_to': 'none',  # No wandb/tensorboard logging
}

print("\n--- Training Configuration ---")
for key, value in training_config.items():
    print(f"  {key}: {value}")

## Cell 9: Setup Custom Training Callback for Periodic BERTScore Evaluation

In [ ]:
from transformers import TrainerCallback
import torch

# Create metrics CSV file for tracking validation metrics
metrics_csv_path = os.path.join(metrics_dir, 'validation_metrics.csv')

class BERTScoreCheckpointCallback(TrainerCallback):
    """Custom callback to compute BERTScore every 3 epochs."""

    def __init__(self, evaluator, val_dataset, val_languages, metrics_csv_path, bertscore_interval=3):
        self.evaluator = evaluator
        self.val_dataset = val_dataset
        self.val_languages = val_languages
        self.metrics_csv_path = metrics_csv_path
        self.bertscore_interval = bertscore_interval
        self.metrics_history = []
        self.best_val_loss = float('inf')

        # Initialize CSV
        with open(metrics_csv_path, 'w') as f:
            f.write('epoch,train_loss,val_loss,perplexity,bertscore_f1_cebuano,bertscore_f1_ilocano,bertscore_f1_tagalog,bertscore_f1_avg\\n')

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        \"\"\"Called after evaluation step.\"\""\n        if metrics is None:\n            return\n        \n        epoch = int(state.global_step / state.steps_per_epoch) if hasattr(state, 'steps_per_epoch') else 0\n        val_loss = metrics.get('eval_loss', float('inf'))\n        \n        # Compute perplexity\n        perplexity = torch.exp(torch.tensor(val_loss)).item() if val_loss != float('inf') else 0.0\n        \n        # Track best validation loss for early stopping\n        if val_loss < self.best_val_loss:\n            self.best_val_loss = val_loss\n            print(f\"\\n✓ Epoch {epoch}: New best validation loss: {val_loss:.4f} (perplexity: {perplexity:.4f})\")\n        \n        # Compute BERTScore every N epochs\n        bertscore_metrics = {}\n        if epoch % self.bertscore_interval == 0 and epoch > 0:\n            print(f\"\\n⏱️  Computing BERTScore for validation set (epoch {epoch})...\")\n            \n            # Generate predictions on validation set\n            try:\n                val_predictions = generate_predictions_batch(kwargs['model'], self.val_dataset, kwargs['tokenizer'])\n                \n                # Compute BERTScore for each language\n                for lang in ['cebuano', 'ilocano', 'tagalog']:\n                    lang_indices = [i for i, l in enumerate(self.val_languages) if l == lang]\n                    \n                    if lang_indices:\n                        # Get predictions and references for this language\n                        lang_preds = [val_predictions[i] for i in lang_indices]\n                        lang_refs = [self.val_dataset[i]['text'] for i in lang_indices]\n                        \n                        # Compute similarity scores\n                        scores = []\n                        for pred, ref in zip(lang_preds, lang_refs):\n                            score = self.evaluator.compute_weighted_similarity(pred, ref, lang)\n                            scores.append(score)\n                        \n                        avg_score = np.mean(scores) if scores else 0.0\n                        bertscore_metrics[f'bertscore_f1_{lang}'] = avg_score\n                        print(f\"  {lang.capitalize()}: {avg_score:.4f}\")\n                    else:\n                        bertscore_metrics[f'bertscore_f1_{lang}'] = 0.0\n                \n                avg_bertscore = np.mean(list(bertscore_metrics.values()))\n                bertscore_metrics['bertscore_f1_avg'] = avg_bertscore\n                print(f\"  Average BERTScore: {avg_bertscore:.4f}\")\n                \n            except Exception as e:\n                print(f\"  Warning: Could not compute BERTScore: {e}\")\n                bertscore_metrics = {\n                    'bertscore_f1_cebuano': 0.0,\n                    'bertscore_f1_ilocano': 0.0,\n                    'bertscore_f1_tagalog': 0.0,\n                    'bertscore_f1_avg': 0.0\n                }\n        else:\n            bertscore_metrics = {\n                'bertscore_f1_cebuano': 0.0,\n                'bertscore_f1_ilocano': 0.0,\n                'bertscore_f1_tagalog': 0.0,\n                'bertscore_f1_avg': 0.0\n            }\n        \n        # Get training loss\n        train_loss = metrics.get('loss', 0.0)\n        \n        # Save to CSV\n        with open(self.metrics_csv_path, 'a') as f:\n            f.write(f\"{epoch},{train_loss:.4f},{val_loss:.4f},{perplexity:.4f},\"\n                   f\"{bertscore_metrics['bertscore_f1_cebuano']:.4f},\"\n                   f\"{bertscore_metrics['bertscore_f1_ilocano']:.4f},\"\n                   f\"{bertscore_metrics['bertscore_f1_tagalog']:.4f},\"\n                   f\"{bertscore_metrics['bertscore_f1_avg']:.4f}\\n\")\n        \n        self.metrics_history.append({\n            'epoch': epoch,\n            'train_loss': train_loss,\n            'val_loss': val_loss,\n            'perplexity': perplexity,\n            **bertscore_metrics\n        })\n\ndef generate_predictions_batch(model, dataset, tokenizer, batch_size=8, max_length=100):\n    \"\"\"Generate predictions for validation/test set in batches.\"\"\"\n    model.eval()\n    predictions = []\n    \n    with torch.no_grad():\n        for i in range(0, len(dataset), batch_size):\n            batch = dataset[i:i+batch_size]\n            texts = batch['text'] if 'text' in batch else batch['conversations']\n            \n            # Generate predictions\n            try:\n                inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device)\n                outputs = model.generate(**inputs, max_new_tokens=max_length, temperature=1.0, top_p=0.95)\n                batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)\n                predictions.extend(batch_preds)\n            except Exception as e:\n                print(f\"Warning in batch prediction: {e}\")\n                predictions.extend(texts)  # Fallback to input text\n    \n    return predictions\n\nprint(\"✓ BERTScore callback and prediction functions initialized!\")"

## Cell 10: ⚠️ RUN TRAINING with Periodic BERTScore Evaluation (Takes 30-60 minutes)

In [ ]:
print("\nStarting training with validation monitoring and periodic BERTScore evaluation...")
print("="*60)
print("⏱️  This will take 30-60 minutes depending on your data size...")
print("📊 Fast metrics (loss/perplexity) computed EVERY epoch")
print("🎯 Slow metrics (BERTScore) computed EVERY 3 epochs")
print("="*60)

# Train model
trainer_stats = trainer.train()

print("\n" + "="*60)
print("✓ Training completed!")
print("="*60)
print(f"Training time: {trainer_stats.metrics['train_runtime']/60:.2f} minutes")
print(f"Total steps: {trainer_stats.metrics.get('train_steps', 'N/A')}")
print(f"\n✓ Validation metrics saved to: {metrics_csv_path}")
print(f"✓ Validation metrics include: loss, perplexity, BERTScore (every 3 epochs)")

## Cell 11: Save Best Model

In [ ]:
# Save best model
print("Saving best model...")
best_model_path = os.path.join(best_model_dir, 'final_model')
model.save_pretrained(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f"✓ Best model saved to {best_model_path}")

# Get training statistics
train_history = trainer.state.log_history
print(f"\n--- Training Statistics ---")
print(f"  Final training loss: {train_history[-1].get('loss', 'N/A') if train_history else 'N/A'}")
eval_losses = [h.get('eval_loss') for h in train_history if 'eval_loss' in h]
if eval_losses:
    print(f"  Final validation loss: {eval_losses[-1]:.4f}")
    print(f"  Best validation loss: {min(eval_losses):.4f}")

## Cell 12: Convert Best Model to GGUF Format (Q4_K_M)

In [ ]:
print("Converting model to GGUF format with Q4_K_M quantization...")
print("This may take 10-15 minutes...\n")

# Create GGUF output directory
gguf_output_dir = os.path.join(BASE_DRIVE_PATH, 'gguf_model')
os.makedirs(gguf_output_dir, exist_ok=True)

print(f"GGUF output directory: {gguf_output_dir}\n")

# Convert to GGUF with Q4_K_M quantization
try:
    model.save_pretrained_gguf(
        gguf_output_dir,
        tokenizer,
        quantization_method="Q4_K_M",  # Q4_K_M quantization
    )
    print("✓ GGUF conversion completed successfully!")

    # List generated files
    gguf_files = os.listdir(gguf_output_dir)
    print(f"\n--- Generated GGUF Files ---")
    for file in gguf_files:
        file_path = os.path.join(gguf_output_dir, file)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  {file} ({file_size_mb:.2f} MB)")

except Exception as e:
    print(f"⚠️ Error during GGUF conversion: {e}")
    print("Attempting fallback with Q8_0 quantization...")
    try:
        model.save_pretrained_gguf(
            gguf_output_dir,
            tokenizer,
            quantization_method="Q8_0",
        )
        print("✓ Fallback to Q8_0 successful!")
    except Exception as e2:
        print(f"Error with Q8_0 as well: {e2}")

## Cell 13: GGUF Model Usage Instructions

In [ ]:
print("="*70)
print("📁 GGUF MODEL STORAGE & USAGE INSTRUCTIONS")
print("="*70)

print(f"\n✓ GGUF Model Location:")
print(f"  Google Drive: {gguf_output_dir}")
print(f"\n✓ Model File: gemma-3-4b-it.Q4_K_M.gguf (quantized with Q4_K_M)")

print("\n" + "="*70)
print("🚀 HOW TO USE THE GGUF MODEL")
print("="*70)

print("""
1. **Using llama.cpp (Recommended for local inference):**
   - Download: https://github.com/ggerganov/llama.cpp
   - Download the GGUF file from Google Drive
   - Run: ./main -m gemma-3-4b-it.Q4_K_M.gguf -p "Your prompt here"

2. **Using Ollama:**
   - Create Modelfile:
     FROM ./gemma-3-4b-it.Q4_K_M.gguf
   - Install Ollama: https://ollama.ai
   - Run: ollama create gemma3-multilingual -f Modelfile
   - Inference: ollama run gemma3-multilingual "Your prompt"

3. **Using Python (with llama-cpp-python):**
   - Install: pip install llama-cpp-python
   - Load:
     from llama_cpp import Llama
     llm = Llama(model_path="gemma-3-4b-it.Q4_K_M.gguf")
     response = llm("Your prompt")

4. **Model Properties:**
   - Format: GGUF (GPU-friendly, CPU-compatible)
   - Quantization: Q4_K_M (4-bit, ~8-9 GB VRAM / ~4 GB RAM)
   - Base Model: Gemma-3 4B
   - Languages: Cebuano, Ilocano, Tagalog
   - Max Context: 2048 tokens

5. **Download from Google Drive:**
   - Auto-saved to: MyDrive/gemma_3_multilingual/gguf_model/
   - Sync with Google Drive desktop for automatic download
""")

print("="*70)
print("✓ Model is ready for deployment!\n")

## Cell 14: Evaluate Test Set with BERTScore and Compare Baseline

In [ ]:
print("Evaluating on test set with BERTScore computation...\n")

# Load best model for evaluation
best_model_for_eval = FastLanguageModel.from_pretrained(
    model_name=os.path.join(best_model_dir, 'final_model'),
    max_seq_length=2048,
    load_in_4bit=True,
)
best_model_for_eval = FastLanguageModel.for_inference(best_model_for_eval)

# Detect test set languages
test_languages_list = [detect_language_from_text(ex.get('text', '')) for ex in test_dataset]

# Generate test predictions
print("Generating predictions on test set...")
test_predictions = generate_predictions_batch(best_model_for_eval, test_dataset, tokenizer)
print(f"✓ Generated {len(test_predictions)} test predictions\n")

# Compute BERTScore for each language on test set
test_bertscore_results = {
    'language': [],
    'count': [],
    'bertscore_f1': [],
    'bertscore_precision': [],
    'bertscore_recall': [],
}

print("--- Test Set BERTScore Evaluation by Language ---\n")

for lang in ['cebuano', 'ilocano', 'tagalog']:
    lang_indices = [i for i, l in enumerate(test_languages_list) if l == lang]

    if lang_indices:
        # Get predictions and references for this language
        lang_preds = [test_predictions[i] for i in lang_indices]
        lang_refs = [test_dataset[i]['text'] for i in lang_indices]

        # Compute BERTScore
        scores_f1 = []
        for pred, ref in zip(lang_preds, lang_refs):
            score = evaluator.compute_weighted_similarity(pred, ref, lang)
            scores_f1.append(score)

        avg_f1 = np.mean(scores_f1) if scores_f1 else 0.0

        test_bertscore_results['language'].append(lang)
        test_bertscore_results['count'].append(len(lang_indices))
        test_bertscore_results['bertscore_f1'].append(avg_f1)
        test_bertscore_results['bertscore_precision'].append(avg_f1)  # Approximate
        test_bertscore_results['bertscore_recall'].append(avg_f1)     # Approximate

        print(f"{lang.capitalize():12} | Count: {len(lang_indices):4} | F1: {avg_f1:.4f}")
    else:
        print(f"{lang.capitalize():12} | Count: 0 | F1: N/A")

# Create results DataFrame
test_results_df = pd.DataFrame({
    'language': test_bertscore_results['language'],
    'count': test_bertscore_results['count'],
    'test_bertscore_f1': test_bertscore_results['bertscore_f1'],
})

print(f"\n--- Test Set Language Breakdown ---")
print(test_results_df.to_string(index=False))

# Save test results
test_results_path = os.path.join(metrics_dir, 'test_set_breakdown.csv')
test_results_df.to_csv(test_results_path, index=False)
print(f"\n✓ Test results saved to {test_results_path}")

# Compute overall test metrics
overall_f1 = np.mean(test_bertscore_results['bertscore_f1']) if test_bertscore_results['bertscore_f1'] else 0.0
print(f"\n--- Overall Test Set Metrics ---")
print(f"Average BERTScore F1: {overall_f1:.4f}")
print(f"Total test samples: {len(test_dataset)}")

## Cell 15: Compare Fine-tuned Model with Baseline (Optional)

In [ ]:
print("\n" + "="*70)
print("📊 OPTIONAL: Compare Fine-tuned Model with Baseline")
print("="*70)
print("\nTo use this cell, you need a baseline model (e.g., original Gemma-3)")
print("Instructions:")
print("1. Load baseline Gemma-3 model")
print("2. Generate predictions on test set")
print("3. Compute BERTScore for baseline")
print("4. Compare results")
print("\nBaseline comparison code:")
print("-" * 70)

baseline_comparison_code = """
# 1. Load baseline model
print("Loading baseline Gemma-3 model...")
baseline_model, baseline_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it",
    max_seq_length=2048,
    load_in_4bit=True,
)
baseline_model = FastLanguageModel.for_inference(baseline_model)

# 2. Generate baseline predictions
print("Generating baseline predictions...")
baseline_predictions = generate_predictions_batch(baseline_model, test_dataset, baseline_tokenizer)

# 3. Compute baseline BERTScore
print("\\nComputing baseline BERTScore...")
baseline_bertscore_results = {
    'language': [],
    'bertscore_f1': [],
}

for lang in ['cebuano', 'ilocano', 'tagalog']:
    lang_indices = [i for i, l in enumerate(test_languages_list) if l == lang]

    if lang_indices:
        lang_preds = [baseline_predictions[i] for i in lang_indices]
        lang_refs = [test_dataset[i]['text'] for i in lang_indices]

        scores = [evaluator.compute_weighted_similarity(p, r, lang) for p, r in zip(lang_preds, lang_refs)]
        baseline_bertscore_results['language'].append(lang)
        baseline_bertscore_results['bertscore_f1'].append(np.mean(scores) if scores else 0.0)

# 4. Compare results
print("\\n\" + \"=\"*70)
print(\"📈 BASELINE vs FINE-TUNED COMPARISON\")
print(\"=\"*70, end=\"\\n\\n\")

comparison_df = pd.DataFrame({
    'Language': baseline_bertscore_results['language'],
    'Baseline F1': baseline_bertscore_results['bertscore_f1'],
    'Fine-tuned F1': test_bertscore_results['bertscore_f1'][:len(baseline_bertscore_results['language'])],
})

comparison_df['Improvement'] = comparison_df['Fine-tuned F1'] - comparison_df['Baseline F1']
comparison_df['Improvement %'] = (comparison_df['Improvement'] / comparison_df['Baseline F1'] * 100).round(2)

print(comparison_df.to_string(index=False))

baseline_avg_f1 = np.mean(baseline_bertscore_results['bertscore_f1'])
finetuned_avg_f1 = np.mean(test_bertscore_results['bertscore_f1'])
overall_improvement = finetuned_avg_f1 - baseline_avg_f1
overall_improvement_pct = (overall_improvement / baseline_avg_f1 * 100)

print(f\"\\n\" + \"-\"*70)
print(f\"Baseline Average F1:     {baseline_avg_f1:.4f}\")
print(f\"Fine-tuned Average F1:   {finetuned_avg_f1:.4f}\")
print(f\"Overall Improvement:     {overall_improvement:+.4f} ({overall_improvement_pct:+.2f}%)\")
print(\"-\"*70)

# Save comparison
comparison_df.to_csv(os.path.join(metrics_dir, 'baseline_comparison.csv'), index=False)
print(f\"\\n✓ Comparison results saved to: {os.path.join(metrics_dir, 'baseline_comparison.csv')}\")
\"\"\"\n\nprint(baseline_comparison_code)\nprint(\"\\n📝 Copy the code above and run it in a new cell to compare with baseline.\")\nprint(\"=\"*70)

## Cell 16: Plot Training Metrics (Loss, Perplexity, BERTScore)

In [ ]:
print("Generating comprehensive training visualizations...\n")

# Load validation metrics from CSV
validation_metrics_df = pd.read_csv(metrics_csv_path)

# 1. Plot training and validation loss + perplexity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Left: Loss curves
ax1.plot(validation_metrics_df['epoch'], validation_metrics_df['train_loss'],
         marker='o', label='Training Loss', linewidth=2, markersize=6)
ax1.plot(validation_metrics_df['epoch'], validation_metrics_df['val_loss'],
         marker='s', label='Validation Loss', linewidth=2, markersize=6)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: Perplexity
ax2.plot(validation_metrics_df['epoch'], validation_metrics_df['perplexity'],
         marker='D', label='Perplexity', linewidth=2, color='green', markersize=6)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Perplexity', fontsize=12)
ax2.set_title('Validation Perplexity', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
loss_plot_path = os.path.join(metrics_dir, 'training_validation_loss.png')
plt.savefig(loss_plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Loss and perplexity curves saved to {loss_plot_path}")
plt.show()
plt.close()

# 2. Plot BERTScore evolution (if available)
if 'bertscore_f1_avg' in validation_metrics_df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))

    # Filter out epochs where BERTScore = 0 (not computed)
    bertscore_data = validation_metrics_df[validation_metrics_df['bertscore_f1_avg'] > 0]

    if len(bertscore_data) > 0:
        ax.plot(bertscore_data['epoch'], bertscore_data['bertscore_f1_cebuano'],
               marker='o', label='Cebuano', linewidth=2, markersize=8)
        ax.plot(bertscore_data['epoch'], bertscore_data['bertscore_f1_ilocano'],
               marker='s', label='Ilocano', linewidth=2, markersize=8)
        ax.plot(bertscore_data['epoch'], bertscore_data['bertscore_f1_tagalog'],
               marker='^', label='Tagalog', linewidth=2, markersize=8)
        ax.plot(bertscore_data['epoch'], bertscore_data['bertscore_f1_avg'],
               marker='D', label='Average', linewidth=3, color='black', markersize=8)

        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('BERTScore F1', fontsize=12)
        ax.set_title('BERTScore Evolution During Training (Every 3 Epochs)', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1.0])

        plt.tight_layout()
        bertscore_plot_path = os.path.join(metrics_dir, 'training_bertscore_evolution.png')
        plt.savefig(bertscore_plot_path, dpi=300, bbox_inches='tight')
        print(f"✓ BERTScore evolution saved to {bertscore_plot_path}")
        plt.show()
    else:
        print("⚠️  No BERTScore data available for plotting (BERTScore computed every 3 epochs)")

    plt.close()
else:
    print("⚠️  BERTScore metrics not available in CSV")

## Cell 17: Language Distribution Visualization

In [ ]:
# Language distribution across splits
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train_languages_list = [languages[i] for i in range(len(train_dataset))]
val_languages_list = [detect_language_from_text(ex.get('text', '')) for ex in val_dataset]
test_languages_list = [detect_language_from_text(ex.get('text', '')) for ex in test_dataset]

splits_data = {
    'Training': train_languages_list,
    'Validation': val_languages_list,
    'Test': test_languages_list
}

for idx, (split_name, langs) in enumerate(splits_data.items()):
    lang_counts = Counter(langs)
    language_names = list(lang_counts.keys())
    counts = list(lang_counts.values())

    colors = plt.cm.Set3(range(len(language_names)))
    axes[idx].bar(language_names, counts, color=colors, edgecolor='black', linewidth=1.5)
    axes[idx].set_title(f'{split_name} Set (n={len(langs)})', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Count', fontsize=11)
    axes[idx].tick_params(axis='x', rotation=45)

    for lang, count in zip(language_names, counts):
        percentage = (count / len(langs)) * 100
        axes[idx].text(lang, count, f'{percentage:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
dist_plot_path = os.path.join(metrics_dir, 'language_distribution.png')
plt.savefig(dist_plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Language distribution saved to {dist_plot_path}")
plt.show()
plt.close()

## Cell 18: Save Comprehensive Training Summary

In [ ]:
# Compute final metrics from CSV
final_metrics = validation_metrics_df.tail(1).iloc[0] if len(validation_metrics_df) > 0 else {}

# Get final BERTScore values
final_bertscore_cebuano = final_metrics.get('bertscore_f1_cebuano', 0.0) if 'bertscore_f1_cebuano' in final_metrics else 0.0
final_bertscore_ilocano = final_metrics.get('bertscore_f1_ilocano', 0.0) if 'bertscore_f1_ilocano' in final_metrics else 0.0
final_bertscore_tagalog = final_metrics.get('bertscore_f1_tagalog', 0.0) if 'bertscore_f1_tagalog' in final_metrics else 0.0
final_bertscore_avg = final_metrics.get('bertscore_f1_avg', 0.0) if 'bertscore_f1_avg' in final_metrics else 0.0

# Get test BERTScore values
test_bertscore_cebuano = test_bertscore_results['bertscore_f1'][0] if len(test_bertscore_results['bertscore_f1']) > 0 else 0.0
test_bertscore_ilocano = test_bertscore_results['bertscore_f1'][1] if len(test_bertscore_results['bertscore_f1']) > 1 else 0.0
test_bertscore_tagalog = test_bertscore_results['bertscore_f1'][2] if len(test_bertscore_results['bertscore_f1']) > 2 else 0.0
test_bertscore_avg = overall_f1

# Create comprehensive summary
summary_text = f"""
GEMMA-3 MULTILINGUAL FINE-TUNING SUMMARY
{'='*70}

DATASET INFORMATION:
  Total examples: {len(dataset)}
  Training set: {len(train_dataset)} ({len(train_dataset)/len(dataset)*100:.1f}%)
  Validation set: {len(val_dataset)} ({len(val_dataset)/len(dataset)*100:.1f}%)
  Test set: {len(test_dataset)} ({len(test_dataset)/len(dataset)*100:.1f}%)

TRAINING CONFIGURATION:
  Model: Gemma-3 4B (unsloth)
  Quantization: 4-bit (during training)
  LoRA rank: 16
  Epochs: {training_config['num_train_epochs']}
  Batch size: {training_config['per_device_train_batch_size']} (per device)
  Learning rate: {training_config['learning_rate']}
  Optimizer: {training_config['optim']}

TRAINING RESULTS:
  Total steps: {trainer_stats.metrics.get('train_steps', 'N/A')}
  Training time: {trainer_stats.metrics['train_runtime']/60:.2f} minutes
  Final training loss: {final_metrics.get('train_loss', 'N/A')}
  Final validation loss: {final_metrics.get('val_loss', 'N/A')}
  Final perplexity: {final_metrics.get('perplexity', 'N/A')}

VALIDATION METRICS (Final Epoch):
  BERTScore F1 - Cebuano:  {final_bertscore_cebuano:.4f}
  BERTScore F1 - Ilocano:  {final_bertscore_ilocano:.4f}
  BERTScore F1 - Tagalog:  {final_bertscore_tagalog:.4f}
  BERTScore F1 - Average:  {final_bertscore_avg:.4f}

TEST SET METRICS (Best Model):
  BERTScore F1 - Cebuano:  {test_bertscore_cebuano:.4f}
  BERTScore F1 - Ilocano:  {test_bertscore_ilocano:.4f}
  BERTScore F1 - Tagalog:  {test_bertscore_tagalog:.4f}
  BERTScore F1 - Average:  {test_bertscore_avg:.4f}

LANGUAGE DISTRIBUTION:
  Detected languages: {', '.join(set(languages))}

VALIDATION METRICS:
  📊 Fast metrics (computed every epoch):
     - Validation loss
     - Perplexity

  🎯 Slow metrics (computed every 3 epochs):
     - BERTScore F1 for each language
     - CSV saved with all metrics

MODEL OUTPUTS:
  1. LoRA Adapters: {best_model_path}

  2. GGUF Format (Q4_K_M Quantized): {gguf_output_dir}
     - File: gemma-3-4b-it.Q4_K_M.gguf
     - Size: ~8-9 GB
     - Use with: llama.cpp, Ollama, llama-cpp-python

  3. Metrics & Logs: {metrics_dir}
     - validation_metrics.csv                (loss, perplexity, BERTScore)
     - test_set_breakdown.csv               (test set BERTScore by language)
     - complete_training_log.csv            (full trainer logs)
     - training_validation_loss.png         (loss and perplexity curves)
     - training_bertscore_evolution.png     (BERTScore over epochs)
     - language_distribution.png            (language breakdown)
     - training_summary.txt                 (this summary)

RECOMMENDED NEXT STEPS:
  1. Review validation_metrics.csv to track training progress
  2. Check BERTScore evolution in training_bertscore_evolution.png
  3. Compare test set metrics with baseline model (optional cell)
  4. Download GGUF file from Google Drive
  5. Use with llama.cpp for CPU/GPU inference
  6. Deploy with Ollama for production

EVALUATION METHODOLOGY:
  ✅ Validation: Language-aware BERTScore using language-specific BERT models
               - Cebuano/Tagalog: DOST-ASTI/bert-ceb-cased
               - Ilocano: bert-base-multilingual-cased

  ✅ Test Set: Final evaluation on held-out test set with best checkpoint

  ✅ Baseline: Optional comparison with baseline Gemma-3 model
              (Use Cell 15 for baseline comparison code)
"""

summary_path = os.path.join(metrics_dir, 'training_summary.txt')
with open(summary_path, 'w') as f:
    f.write(summary_text)

print(summary_text)
print(f"✓ Summary saved to {summary_path}")

## Cell 19: Save Complete Training Log and Summary

In [ ]:
# Save final validation metrics to CSV (already done in callback, but ensure it's complete)
validation_metrics_df = pd.read_csv(metrics_csv_path)
final_metrics_path = os.path.join(metrics_dir, 'complete_training_log.csv')

# Merge with trainer log history
if hasattr(trainer, 'state') and hasattr(trainer.state, 'log_history'):
    trainer_log_df = pd.DataFrame(trainer.state.log_history)
    # Keep validation metrics as primary source
    validation_metrics_df.to_csv(final_metrics_path, index=False)
    print(f"✓ Complete training log (validation metrics) saved to {final_metrics_path}")

    # Display metrics summary
    print("\n--- Training Metrics Summary ---")
    print(f"Total epochs: {len(validation_metrics_df)}")
    print(f"Lowest validation loss: {validation_metrics_df['val_loss'].min():.4f}")
    print(f"Highest average BERTScore: {validation_metrics_df['bertscore_f1_avg'].max():.4f}")

    print("\n--- Last 5 Epochs ---")
    print(validation_metrics_df.tail(5).to_string(index=False))

print("\n" + "="*70)
print("✅ FINE-TUNING PIPELINE COMPLETED SUCCESSFULLY!")
print("="*70)
print(f"\nAll outputs saved to: {BASE_DRIVE_PATH}")
print("\n📁 Output structure:")
print(f"  ├── checkpoints/                          (training checkpoints)")
print(f"  ├── best_model/                           (best LoRA adapter weights)")
print(f"  ├── gguf_model/                           (GGUF Q4_K_M format)")
print(f"  └── metrics/")
print(f"      ├── validation_metrics.csv            ✅ FAST + SLOW metrics")
print(f"      ├── test_set_breakdown.csv            ✅ Test BERTScore")
print(f"      ├── training_validation_loss.png      ✅ Loss/perplexity curves")
print(f"      ├── training_bertscore_evolution.png  ✅ BERTScore over epochs")
print(f"      ├── language_distribution.png")
print(f"      ├── training_summary.txt")
print(f"      └── complete_training_log.csv")

print("\n🎯 Validation Approach:")
print("  ✅ FAST metrics (computed every epoch):")
print("     - Validation loss")
print("     - Perplexity")
print("  ✅ SLOW metrics (computed every 3 epochs):")
print("     - BERTScore F1 for each language")
print("  ✅ Final TEST set evaluation:")
print("     - BERTScore on best checkpoint")
print("     - Language-specific scores")

print("\n📊 Key Files for Review:")
print(f"  1. {metrics_csv_path}")
print(f"     → Validation metrics (loss, perplexity, BERTScore every 3 epochs)")
print(f"  2. {os.path.join(metrics_dir, 'training_validation_loss.png')}")
print(f"     → Training/validation loss and perplexity curves")
print(f"  3. {os.path.join(metrics_dir, 'training_bertscore_evolution.png')}")
print(f"     → BERTScore evolution during training")

## Cell 20: [Optional] Test Model Inference

In [ ]:
# Save final validation metrics to CSV
if hasattr(trainer, 'state') and hasattr(trainer.state, 'log_history'):
    final_metrics_df = pd.DataFrame(trainer.state.log_history)
    final_metrics_path = os.path.join(metrics_dir, 'complete_training_log.csv')
    final_metrics_df.to_csv(final_metrics_path, index=False)
    print(f"✓ Complete training log saved to {final_metrics_path}")

    # Display first few rows
    print("\n--- Training Log Sample ---")
    print(final_metrics_df.head(10).to_string())

print("\n" + "="*60)
print("✅ FINE-TUNING PIPELINE COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"\nAll outputs saved to: {BASE_DRIVE_PATH}")
print("\n📁 Output structure:")
print(f"  ├── checkpoints/          (training checkpoints)")
print(f"  ├── best_model/           (LoRA adapter weights)")
print(f"  ├── gguf_model/           (GGUF Q4_K_M format) ⭐ RECOMMENDED")
print(f"  └── metrics/              (all metrics and visualizations)")
print("\n🎯 Your trained multilingual model is ready!")
print("\n📊 Model Details:")
print(f"  - GGUF File: gemma-3-4b-it.Q4_K_M.gguf (Q4_K_M quantized)")
print(f"  - Location: {gguf_output_dir}")
print(f"  - Use with: llama.cpp, Ollama, or llama-cpp-python")
print(f"  - Languages: Cebuano, Ilocano, Tagalog")
print("\n📥 Next steps:")
print("1. Download GGUF file from Google Drive")
print("2. Use with llama.cpp: ./main -m gemma-3-4b-it.Q4_K_M.gguf (CPU/GPU)")
print("3. Or use with Ollama for production deployment")